[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239232-lesson-6-agent)

# 智能代理

## 回顾

我们构建了一个路由器。

* 我们的聊天模型会根据用户输入决定是否进行工具调用
* 我们使用条件边来路由到一个调用我们工具的节点或简单地结束

![Screenshot 2024-08-21 at 12.44.33 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac0ba0bd34b541c448cc_agent1.png)

## 目标

现在，我们可以将其扩展为一个通用的代理架构。

在上述路由器中，我们调用了模型，如果它选择调用工具，我们向用户返回一个`ToolMessage`。
 
但是，如果我们简单地将该`ToolMessage`*传回给模型*呢？

我们可以让它要么（1）调用另一个工具，要么（2）直接响应。

这就是[ReAct](https://react-lm.github.io/)背后的直觉，这是一个通用的代理架构。
  
* `act`（行动）- 让模型调用特定工具
* `observe`（观察）- 将工具输出传回模型
* `reason`（推理）- 让模型对工具输出进行推理，以决定下一步做什么（例如，调用另一个工具或直接响应）

这种[通用架构](https://blog.langchain.dev/planning-for-agents/)可以应用于许多类型的工具。

![Screenshot 2024-08-21 at 12.45.43 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac0b4a2c1e5e02f3e78b_agent2.png)

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

在这里，我们将使用[LangSmith](https://docs.smith.langchain.com/)进行[追踪](https://docs.smith.langchain.com/concepts/tracing)。

我们将记录到一个项目`langchain-academy`。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """将a和b相乘。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将是一个工具
def add(a: int, b: int) -> int:
    """将a和b相加。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """将a除以b。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatOpenAI(model="gpt-4o")

# 对于这个ipynb，我们将并行工具调用设置为false，因为数学通常是按顺序完成的，这次我们有3个可以做数学的工具
# OpenAI模型特别默认为并行工具调用以提高效率，参见 https://python.langchain.com/docs/how_to/tool_calling_parallel/
# 试着玩一玩，看看模型如何处理数学方程式！
llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)

让我们创建我们的LLM并用整体期望的代理行为来提示它。

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="你是一个有用的助手，负责对一组输入执行算术运算。")

# 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

如前所述，我们使用`MessagesState`并定义一个带有我们工具列表的`Tools`节点。

`Assistant`节点只是我们绑定了工具的模型。

我们创建一个带有`Assistant`和`Tools`节点的图。

我们添加`tools_condition`边，它根据`Assistant`是否调用工具来路由到`End`或`Tools`。

现在，我们添加一个新步骤：

我们将`Tools`节点连接*回*到`Assistant`，形成一个循环。

* 在`assistant`节点执行后，`tools_condition`检查模型的输出是否是工具调用。
* 如果是工具调用，流程被定向到`tools`节点。
* `tools`节点连接回`assistant`。
* 只要模型决定调用工具，这个循环就会继续。
* 如果模型响应不是工具调用，流程被定向到END，终止过程。

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些执行工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些决定控制流如何移动
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自assistant的最新消息（结果）是工具调用 -> tools_condition路由到tools
    # 如果来自assistant的最新消息（结果）不是工具调用 -> tools_condition路由到END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# 显示
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="将3和4相加。将输出乘以2。将输出除以5")]
messages = react_graph.invoke({"messages": messages})

In [ ]:
for m in messages['messages']:
    m.pretty_print()

## LangSmith

我们可以在LangSmith中查看追踪。